In [1]:
from glob import glob
from langchain_classic import text_splitter
from langchain_core import documents
from openai.types import vector_store

for g in glob('./data/*.csv'):
    print(g)

./data\detail_policy2_utf8.csv


In [3]:
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF 파일을 읽고 청크 단위로 텍스트를 잘라 리스트로 담아 반환
def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunck_overlap=100):
    print(f'PDF: {pdf_path} ------')

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunck_overlap
    )

    splits = text_splitter.split_documents(data_from_pdf)

    print(f'Number of splits: {len(splits)}\n')

    return splits

# CSV 파일을 읽고 청크 단위로 텍스트를 잘라 리스트로 담아 반환
def read_csv_and_split_text(csv_path, chunk_size=1000, chunck_overlap=100):
    print(f'CSV: {csv_path} ------')

    csv_loader = CSVLoader(csv_path, encoding="utf-8")
    data_from_csv = csv_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunck_overlap
    )

    splits = text_splitter.split_documents(data_from_csv)

    print(f'Number of splits: {len(splits)}\n')

    return splits

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

# vectorstore 설정
embedding = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1",
    model="text-embedding-bge-m3-ko",
    check_embedding_ctx_length=False
)

persist_directory='./chroma_store'

if os.path.exists(persist_directory):
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding
    )
else:
    print("Creating new Chroma Store")

    vectorstore = None
    for g in glob('./data/*.csv'):
        chunks = read_csv_and_split_text(g)
        # 100개씩 나눠서 저장
        for i in range(0, len(chunks), 100):
            if vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=chunks[i:i+100],
                    embedding=embedding,
                    persist_directory=persist_directory
                )
            else:
                vectorstore.add_documents(
                    documents=chunks[i:i+100]
                )

Creating new Chroma Store
CSV: ./data\detail_policy2_utf8.csv ------
Number of splits: 42



In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

chunks = retriever.invoke("현금지급 지원")

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)

{'source': './data\\detail_policy2_utf8.csv', 'row': 7}
피부양가족 보조금 지원(22만원/월)
중증 후유장애인 및 유자녀 장학금 지원(분기 초등학생 25만원, 중학생 35만원, 고등학생 45만원/분기
유자녀 자립지원금: 월7만 / 유자녀
유자녀 무이자 생활자금 대출 지원(25만원/월)

2. 정서적 지원

심리상담 및 트라우마PTSD 치료지원
피해자 방문돌봄 및 생활지원
유자녀 학습진로 지원
간병간호 및 응급처치 교육 지원
주거환경 개선 및 응급안전 스마트홈 조성 지원
※ 지원내용 및 지원금액은 지원대상별 기준에 따라 상이할 수 있음
신청절차: 1. 거주지 읍/면/동 주민센터, 자동차손해배상진흥원에서 ‘서비스 신청’
2. 담당 시/군/구청 또는 자동차손해배상진흥원에서 조사 및 심사
3. 담당 시/군/구청 또는 자동차손해배상진흥원에서 보장 결정
4. 담당 시/군/구청 또는 자동차손해배상진흥원에서 대상자에게 서비스 제공
5. 담당 시/군/구청 또는 자동차손해배상진흥원에서 서비스 제공 이후 대상자의 상황 관리
문의처목록: 자동차사고 피해자 지원사업 콜센터: 1544-0049
홈페이지목록: 자동차사고 피해자 지원사업 홈페이지: https://tvsis.tacss.or.kr/tvsis/main.do
근거법령목록: 자동차손해배상 보장법
{'row': 16, 'source': './data\\detail_policy2_utf8.csv'}
서비스ID: WLF00006264
서비스명: 생활안정자금(융자)(이차보전)
소관부처명: 고용노동부 퇴직연금복지과
서비스요약: 근로자, 특수형태근로종사자, 1인 자영업자가 혼례·자녀 양육 등을 대출할 대, 대출 이자의 일부를 근로복지공단이 지원해 금융부담을 완화합니다.
기준연도: 2026
문의처: 1588-0075
지원주기: 1회성
제공유형: 기타
생애주기: 청년, 노년, 중장년
관심주제: 일자리
가구유형: 저소득, 다자녀
대상자상세내용: 근로자, 특수형태근로종사자, 1인 자영업자(산재보험 가입 중

여기까지가 벡터 DB 생성

In [9]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    model="google/gemma-4-e2b",
    api_key="lm-studio",
    temperature=0.7
)
model.invoke('안녕하세요!')

AIMessage(content='안녕하세요! 만나서 반갑습니다. 😊\n\n저는 사용자님의 질문에 답하고, 정보를 제공하며, 다양한 작업을 도와드릴 수 있는 AI 어시스턴트입니다.\n\n궁금한 점이 있으시거나 도움이 필요하시면 언제든지 말씀해주세요! 잘 부탁드립니다. 😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 18, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'google/gemma-4-e2b', 'system_fingerprint': 'google/gemma-4-e2b', 'id': 'chatcmpl-gzc9fhxlxpldqa9xhejsl8', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f1cd7-8fd9-7450-808e-2168b6cc3b1b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 57, 'total_tokens': 75, 'input_token_details': {}, 'output_token_details': {'reasoning': 0}})

In [7]:
# 라우터 설정
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal
from pydantic import BaseModel, Field

# Data Model
class RouteQuery(BaseModel):
    """사용자 쿼리를 가장 관련성이 높은 데이터 소스로 라우팅"""
    datasource: Literal["vectorstore", "casual_talk"] = Field(
        ...,
        description="""
        사용자 질문에 따라 casual_talk 또는 vectorstore로 라우팅합니다.
        - casual_talk: 일상 대화를 위한 데이터 소스, 사용자가 일상적인 질문을 할 때 사용됨
        - vectorstore: 사용자 질문에 답하기 위해 RAG로 vectorstore 검색이 필요한 경우 사용
        """
    )

In [10]:
# 특정 모델을 구조화된 출력과 함께 사용하기 위해서 설정
structured_llm_router = model.with_structured_output(RouteQuery)

router_system = """
    당신은 사용자의 질문을 vectorstore 또는 casual_talk으로 라우팅하는 전문가입니다.
    - vectorstore에는 복지정책과 관련된 문서가 포함되어 있습니다. 이 주제에 대한 질문에는 vectorstore를 사용하시오.
    - 사용자의 질문이 일상 대화와 관련된 경우 casual_talk을 사용하시오.
"""

# 시스템 메시지와 사용자의 질문을 포함하는 프롬프트 템플릿 설정
route_prompt = ChatPromptTemplate.from_messages([
    ("system", router_system),
    ("human", "{question}"),
])

# 라우터 프롬프트와 구조화된 출력 모델을 결합한 객체
question_router = route_prompt | structured_llm_router

In [ ]:
print(
    question_router.invoke({
        "question": "난 노인인데 내가 받을 수 있는 복지정책을 알려줘."
    })
)

print(
    question_router.invoke({
        "question": "잘 지냈어?"
    })
)

## 랭그래프로 RAG 에이전트 만들기

In [22]:
from langchain_core.prompts import PromptTemplate

class GradeDocuments(BaseModel):
    """검색된 문서가 질문과 관련성 있는 지 yes, no로 평가한다."""
    binary_score: Literal["yes", "no"] = Field(
        description="문서가 질문과 관련이 있는지 여부를 'yes' 또는 'no' 로 평가합니다."
    )

structured_llm_grader = model.with_structured_output(GradeDocuments)

In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

def expand_query(query: str, n: int = 3) -> list[str]:
    """LLM으로 쿼리를 확장합니다."""
    llm = ChatOpenAI(
        base_url="http://localhost:1234/v1",
        model="google/gemma-4-e2b",
        api_key="lm-studio",
        temperature=0.7
    )
    prompt = ChatPromptTemplate.from_template(
        "다음 질문에 대해 의미는 같지만 표현이 다른 검색 쿼리를 "
        "{n}개 생성하세요. 각 쿼리를 줄바꿈으로 구분하세요.\n\n"
        "원본 질문: {query}"
    )
    response = llm.invoke(prompt.format(query=query, n=n))
    expanded = response.content.strip().split("\n")
    return [q.strip() for q in expanded if q.strip()][:n]

In [25]:
# 기존 대화 내용이 계속 이어질 필요가 없는 경우: PromptTemplate
# 이어질 필요가 있는 경우: ChatPromptTemplate
grader_prompt = ChatPromptTemplate.from_template("""
    당신은 검색된 문서가 사용자 질문과 관련이 있는지 평가하는 평가자입니다.\n
    문서에 사용자 질문과 관련된 키워드 또는 의미가 포함되어 있으면, 해당 문서를 관련성이 있다고 평가하십시오.\n
    엄격한 테스트가 필요하지 않습니다. 목표는 잘못된 검색 결과를 걸러내는 것입니다.\n
    문서가 질문과 관련이 있는 지 여부를 나타내기 위해 'yes' 또는 'no'로 이진 점수를 부여하십시오.\n
    질문은 4개가 제공될 것이며 4개의 파생 질문 중 하나라도 답변할 수 있다면 'yes'를 출력하시오.

    Retrieved document: \n {document} \n\n
    User question: {question}
""")

retrieval_grader = grader_prompt | structured_llm_grader
question = "법무부와 연관되어있는 정책"
documents = retriever.invoke(question)

for doc in documents:
    print(doc)


page_content='서비스ID: WLF00006308
서비스명: 무료법률상담
소관부처명: 법무부 인권구조과
서비스요약: 경제적으로 어렵거나 법을 잘 몰라 법의 보호를 충분히 받지 못하는 국민을 지원합니다.
기준연도: 2026
문의처: 132
지원주기: 수시
제공유형: 프로그램/서비스(서비스)
생애주기: 청년, 아동, 청소년, 노년, 임신 · 출산, 중장년, 영유아
관심주제: 법률
가구유형: 저소득
대상자상세내용: 법률 상담이 필요한 국민을 대상으로 지원합니다.
선정기준내용: 지원대상의 내용을 참고해주시기 바랍니다.
급여서비스내용: 법률 관련 간담 상단, 손해배상, 계약 등 민사소송, 임대자, 개인회생 및 파산 등 무료법률상담을지원합니다.
신청절차: 1. 거주지 읍/면/동 주민센터, 대한변협 법률구조재단에서 ‘서비스 신청’
2. 거주지 읍/면/동 주민센터, 대한가정법률복지상담원에서 ‘서비스 신청’
3. 거주지 읍/면/동 주민센터, 한국가정법률상담소에서 ‘서비스 신청’
4. 거주지 읍/면/동 주민센터, 대한법률구조공단에서 ‘서비스 신청’
5. 담당 시/군/구청 또는 대한변협 법률구조재단에서 조사 및 심사
6. 담당 시/군/구청 또는 대한가정법률복지상담원에서 조사 및 심사
7. 담당 시/군/구청 또는 한국가정법률상담소에서 조사 및 심사
8. 담당 시/군/구청 또는 대한법률구조공단에서 조사 및 심사
9. 담당 시/군/구청 또는 대한변협 법률구조재단에서 보장 결정
10. 담당 시/군/구청 또는 대한가정법률복지상담원에서 보장 결정
11. 담당 시/군/구청 또는 한국가정법률상담소에서 보장 결정
12. 담당 시/군/구청 또는 대한법률구조공단에서 보장 결정
13. 담당 시/군/구청 또는 대한변협 법률구조재단에서 대상자에게 서비스 제공
14. 담당 시/군/구청 또는 대한가정법률복지상담원에서 대상자에게 서비스 제공
15. 담당 시/군/구청 또는 한국가정법률상담소에서 대상자에게 서비스 제공
16. 담당 시/군/구청 또는 대한법률구조공단에서 대상자에게 서비스 제공' metada

In [ ]:
filtered_docs = []

for i, doc in enumerate(documents):
    print(f"Document {i + 1}")
    queries_string = "\n".join([f"{i+1}. {q}" for i, q in enumerate(expand_query(question))])
    is_relevant = retrieval_grader.invoke({"question": queries_string, "document": doc.page_content})
    print(is_relevant)
    print(doc.page_content[:200])
    print("========================\n\n")

    if is_relevant.binary_score == "yes":
        filtered_docs.append(doc)

print(f"Filtered documents: {len(filtered_docs)}")

Document 1
binary_score='yes'
서비스ID: WLF00006308
서비스명: 무료법률상담
소관부처명: 법무부 인권구조과
서비스요약: 경제적으로 어렵거나 법을 잘 몰라 법의 보호를 충분히 받지 못하는 국민을 지원합니다.
기준연도: 2026
문의처: 132
지원주기: 수시
제공유형: 프로그램/서비스(서비스)
생애주기: 청년, 아동, 청소년, 노년, 임신 · 출산, 중장년, 영유아
관심주제: 


Document 2
binary_score='no'
서비스ID: WLF00006309
서비스명: 법률구조
소관부처명: 법무부 인권구조과
서비스요약: 경제적으로 어렵거나 법률 지식이 부족해 법의 보호를 충분히 받지 못하는 국민의 기본적 인권 옹호를 위해 법률상담, 소송 대리 및 형사변호 등 법률 서비스를 지원합니다.
기준연도: 2026
문의처: 132
지원주기: 수시
제공유형: 프로그램/서비스(서비스)
생애주


Document 3
binary_score='no'
서비스ID: WLF00006310
서비스명: 개인회생 파산 종합지원(지원센터)
소관부처명: 법무부 인권구조과
서비스요약: 감당할 수 없는 빚으로 개인회생, 개인파산 및 면책 제도 이용을 원하는 국민을 지원합니다.
기준연도: 2026
문의처: 132
지원주기: 수시
제공유형: 프로그램/서비스(서비스)
생애주기: 청년, 아동, 청소년, 임신 · 출산, 노년, 


Document 4
